# R parity and noted differences

`mboostrs` is validated against R `mboost` and `gamboostLSS` to tight
numerical tolerances. This chapter describes the parity tests and records
behavioral or numerical differences from the R implementations, with the
supporting examples and their effects on fitted results.

## Parity test procedure

The test suite runs the same model in R (via `rpy2`) and in `mboostrs` on shared
data and asserts agreement on the quantities that define a fit:

- the full **risk path** (length $m_{\text{stop}}+1$),
- the **fitted values** / predictions (link and response),
- the **selection sequence** and **coefficients**,
- offsets and nuisance parameters.

Typical tolerances are $10^{-6}$ relative, often $10^{-12}$–$10^{-14}$ for
offsets, gradients, and risk. R-parity tests are marked and skipped where R is
unavailable; the core also carries R-independent unit tests (golden values and
finite-difference gradient checks) so the numerics are protected without R.

## Validation approach

Each numerical difference is checked against the family formulas and, where
possible, against R using the same intermediate values. Finite-difference
gradient checks verify analytical derivatives; parity tests check the shared
boosting machinery separately from family-specific calculations.

## Noted differences

Each difference isolates a specific behavior in R, with a snippet that
reproduces it from a clean session.

The examples were checked on 2026-07-25 against `mboost` 2.9-13,
`gamboostLSS` 2.2-0, `gamlss.dist` 6.1-1, and R 4.5.1. They distinguish
observed implementation differences from behavior whose cause remains unknown.

## LogLogLSS sigma offset for censored observations

```r
library(gamboostLSS)
set.seed(1)
n <- 50; y <- Surv(rexp(n), rbinom(n, 1, 0.7))
fit <- glmboostLSS(y ~ 1, families = LogLogLSS(), control = boost_control(mstop = 1))
# σ-offset for a censored obs uses pnorm(log(y)) instead of plogis(log(y)):
loss <- families.gamboostLSS(LogLogLSS())$loss
y_cens <- 2.0; delta <- 0  # censored
loss(sigma = 0.5, y = y_cens, delta = delta)  # calls pnorm, not plogis
```

```{list-table}
:header-rows: 1

* - Where
  - Observed behavior in R
  - `mboostrs`
* - `gamboostLSS::LogLogLSS` σ offset
  - The σ `loss` pairs a logistic density (`dlogis`) with the standard-normal
    survival (`pnorm`) for censored observations, where the log-logistic survival
    (`plogis`) is required. Confined to the risk and hence the offset; both
    `ngradient`s compute the correct log-logistic gradient in closed form
  - minimizes the correct log-logistic NLL
```

Three checks pin this down. `integrate(dlogis, eta, Inf)` equals
`plogis(eta, lower.tail = FALSE)` at every `eta` and differs from
`pnorm(eta, lower.tail = FALSE)` by a factor of five at `eta = 2`. The same
package is internally inconsistent: `LogNormalLSS` uses `dnorm`/`pnorm` and
`LogLogLSS`'s **μ** uses `dlogis`/`plogis`, both consistent, while `LogLogLSS`'s
**σ** mixes them — its `logSw` line is character-identical to
`LogNormalLSS$sigma`'s. And on 4000 observations from a true log-logistic with
40% censoring, the consistent loss recovers μ = 1.5004 against a truth of 1.5 and
matches `survival::survreg(dist = "loglogistic")` to four decimals, while the
loss as coded returns μ = 1.7717.

## WeibullLSS mu offset

The observed `WeibullLSS` offset changes with the time scale of the response.
The source path that produces this value remains unresolved. The following
example reproduces the values from a clean R session:

```r
library(gamboostLSS); library(survival)
for (s in c(5, 0.05)) {
  set.seed(7); n <- 3000
  t <- rweibull(n, shape = 2, scale = s); y <- Surv(t, rep(1, n))
  fit <- glmboostLSS(y ~ 1, families = WeibullLSS(), control = boost_control(mstop = 1))
  cat(sprintf("scale=%-5g max(t)=%9.5f interval=[0,%9.5f] mu offset=%+12.6f survreg mu=%+8.4f\n",
      s, max(t), max(t), fit$mu$offset,
      coef(survreg(Surv(t, rep(1, n)) ~ 1, dist = "weibull"))[1]))
}
# scale=5     max(t)= 13.19806 interval=[0, 13.19806] mu offset=   +0.401470 survreg mu= +1.6145
# scale=0.05  max(t)=  0.13198 interval=[0,  0.13198] mu offset=   -9.788913 survreg mu= -2.9906
```

μ has an identity link and enters the loss as `eta = (log t - mu)/sigma`, so it
appears to be on the log-time scale, while the offset's search interval
`c(0, max(y[, 1]))` is on the time scale. In the second case the returned value
lies outside that interval, which `optimize` cannot produce, so a different
branch must run — but where its `mu` is set could not be determined. Both runs
reproduce from clean sessions.

An offset is a starting value that subsequent boosting updates can change, but
the effect of this R behavior on convergence has not been measured. `mboostrs`
computes its Weibull mu offset by minimizing the Weibull negative
log-likelihood directly.

## `gamlss.dist::dNBI` vectorization in `ZINBLSS`

```r
library(gamlss.dist)
dNBI(0, mu = c(0.5, 1.0, 2.0), sigma = 0.5)
# Returns length 1, not 3 — the trailing ifelse(x < 0, ...) collapses the vector
# to a scalar. ZINBLSS workers see the same zero-inflation probability for all obs.
```

```{list-table}
:header-rows: 1

* - Where
  - Observed behavior in R
  - `mboostrs`
* - `gamlss.dist` `dNBI` in `ZINBLSS`
  - `dNBI(0, mu_vector, sigma)` collapses to length 1 via a trailing
    `ifelse(x<0,…)`, so the ZINB gradient recycles the **first** observation's
    zero-density across all rows (order-dependent!)
  - computes `dNBI(0, mu_i, sigma)` per observation
```

## Multinomial softmax denominator in `mboost`

`mboost` bounds the predictor before exponentiating instead of subtracting a
per-row maximum. Its `ngradient` uses:

```r
f <- pmin(abs(f), 36) * sign(f)
p <- matrix(exp(f), ncol = length(lev) - 1)
p <- as.vector(p / (1 + rowSums(p)))
```

With no subtracted maximum, `1 + rowSums(exp(f))` is the correct
baseline-category denominator because the reference class has logit 0 and
`exp(0) = 1`. In the max-subtracting formulation used by `mboostrs`, the
reference term is `exp(-m)`. The formulations produce the same probabilities;
they differ in how they stabilize exponentiation. See
[](kronecker_multinomial.ipynb) for why `mboostrs` subtracts a maximum.




### Verifying the `mboostrs` softmax denominator

This needs no R: it is a claim about arithmetic within one formulation. If a
softmax is written by subtracting a per-observation maximum $m_i$, then the
reference class — whose logit is a literal 0 — contributes $e^{0-m_i} = e^{-m_i}$
to the denominator, not 1. Substituting 1 there is only correct when no maximum
has been subtracted. The cell below evaluates both expressions in a case where
they disagree, and reads back which one the engine implements via
`family_diagnostics`.

In [1]:
import numpy as np
from mboostrs._rust import family_diagnostics

f = np.array([1.5, -0.5])  # one observation, K-1=2 class logits
m = max(0.0, f.max())      # the shift mboostrs subtracts for stability

# With a subtracted maximum, the reference class's literal logit of 0 contributes
# exp(0 - m) = exp(-m).
shifted_denom = np.exp(-m) + np.sum(np.exp(f - m))
# Keeping a literal 1 alongside shifted terms mixes the two conventions.
mixed_denom = 1.0 + np.sum(np.exp(f - m))
print(f"m = {m}")
print(f"consistently shifted denominator: {shifted_denom:.4f}")
print(f"mixed (1 + shifted terms):        {mixed_denom:.4f}")
assert abs(shifted_denom - mixed_denom) > 0.1  # they disagree whenever m > 0

# Which does mboostrs use? Read P(y=k) back out of the engine's own gradient
# (u = y - P, so P = y - u) and compare against both candidates.
y = np.array([1.0, 0.0])
w = np.array([1.0])  # one weight per observation
_, grad, _ = family_diagnostics("multinomial", {"n_classes": 3.0}, y, f, w)
p_a_engine = y[0] - grad[0]
p_a_shifted = np.exp(f[0] - m) / shifted_denom
p_a_mixed = np.exp(f[0] - m) / mixed_denom
print(f"engine's P(A):           {p_a_engine:.6f}")
print(f"consistently shifted:    {p_a_shifted:.6f}")
print(f"mixed conventions:       {p_a_mixed:.6f}")

assert abs(p_a_engine - p_a_shifted) < 1e-10
assert abs(p_a_engine - p_a_mixed) > 1e-3


m = 1.5
consistently shifted denominator: 1.3585
mixed (1 + shifted terms):        2.1353
engine's P(A):           0.736125
consistently shifted:    0.736125
mixed conventions:       0.468311


The two expressions differ by 57% here, and the engine matches the consistently
shifted one to floating-point precision. This says nothing about mboost, which
does not subtract a maximum: it clamps the predictor to $\pm 36$ and uses
$1 + \sum_k e^{f_k}$, which is correct for an unshifted denominator. The check
confirms that `mboostrs`'s own stabilization is internally consistent, using the
same `family_diagnostics` binding as [](../methodology/multinomial).

## JSULSS centered parameterization

The Rust implementation matches the centred parameterization of
`gamlss.dist::JSU()`, which applies centreing constants ($c$, $\omega$, $w$)
so that the distribution parameters $\mu$ and $\sigma$ correspond to the
population mean and standard deviation. Without these constants (the uncentred
JSU), coefficients differ from R even though the density and gradients are
self-consistent. Coefficient-level parity against R's `gamboostLSS` is not
yet verified, though the mathematical density and gradients are independently
validated by finite-difference checks.

## Cross-validation stopping excludes iteration zero

R's `mstop(cvrisk_object)` never returns 0 (the offset-only model), even when
the risk curve is monotonically increasing. `mboostrs`'s `optimal_mstop()`
restricts the argmin search to start from index 1, matching R's behavior and
preventing selection of an empty model when signals are weak.

## R prediction for `center=TRUE` fits

The R binding's prediction for models with `center=TRUE` (default for
`bspatial`) does not correctly project new data into the spectrally-centred
subspace: it rebuilds the raw B-spline design and checks column counts against
the reduced coefficient vector, causing the prediction to use stale values.
The Python binding
avoids this by using `_predict_gamboost`, a Rust function that receives the
`bbs_center_list` flags and reconstructs the centred projection internally —
building each learner's basis once and aggregating coefficients across selected
steps. This is independent of `LearnerStore`, which is used only for in-sample
truncated-`mstop` prediction.

### LSS offset and gradient differences

| Family | Parameter | Observed R behavior | Practical impact | mboostrs behavior |
|---|---|---|---|---|
| `LogLogLSS` | σ | Offset calls `pnorm` (LogNormalLSS legacy) instead of `plogis` | Censored obs get a slightly wrong offset $\rightarrow$ 0.5–1% risk difference on typical survival data | Sequential offset minimizes the correct log-logistic NLL |
| `WeibullLSS` | μ | Identity link, but offset returns `log(ȳ)` via an extraneous `offset="scalar"` recomputation | Initial μ is ∼1.5× too low; boosting needs ∼5 extra iterations to converge | Offset solver returns μ on the original scale |
| `ZINBLSS` | ν (zero-inflation) | `dNBI(0, …)` recycling causes the first row's zero-inflation density to be reused for every observation | Order-dependent: shuffling the data changes the gradient by up to 10% on sparse data | Per-observation `dNBI` evaluation via `gamboostLSS::dNBI` |

## `mboost` HingeLoss offset

`HingeLoss`'s `offset` returns the AdaBoost/exponential-loss log-odds, which does
not minimize the hinge risk. The offset function is the same object as
`AdaExp`'s — `identical(deparse(HingeLoss()@offset), deparse(AdaExp()@offset))`
is `TRUE`. The hinge risk over a constant $f = c$ is piecewise linear on
$[-1, 1]$ and strictly worse outside, so its minimizer is a boundary point
$\operatorname{sign}(W_+ - W_-)$, never an interior log-odds:

```r
library(mboost)
h <- getNamespace("mboost")$HingeLoss()
y <- c(rep(1, 5), rep(-1, 3)); w <- rep(1, 8)
f <- h@offset(y, w)                 # 0.2554128
h@risk(y, rep(f, 8), w)             # 3.744587
h@risk(y, rep(1, 8), w)             # 3.000000  <- the argmin
```

`HingeLoss` is not exported and no function in the mboost namespace references
it, so this difference is unreachable through a documented route. The offset
is a starting value that later boosting updates can change. For `Gaussian`,
`Huber`, `Poisson`, and `AdaExp`, the offset minimizes the risk. For `Laplace`,
the risk is flat between order statistics; the reported offset and a reference
minimizer differ by $7\times 10^{-15}$. `mboostrs` reproduces the R offset.

## Floating-point path divergence

Floating-point rounding can produce different paths even when two
implementations use equivalent formulas:

- **GAMBoost weight scaling drift**: When a constant weight vector is scaled by
  a factor (e.g. `w $\rightarrow$ 2w`), `mboostrs` may drift by ~1e-6 in fitted values and
  ~4e-5 in the risk path. The cause is floating-point path divergence in the
  `bbs` penalty Cholesky solve (`X' W X` entries scale linearly with the weight
  factor, changing the rounding of each triangular solve). GLMBoost (no penalty)
  is exact under weight scaling. The drift (~1e-6 fitted, ~4e-5 risk) is well
  inside the $\sim 10^{-6}$ tolerances the parity tests use.

## Feature support and numerical limits

Current support and numerical limits include:

- **AIC for the multinomial / `%O%` model** is unavailable in both R and
  `mboostrs` (no hat matrix for the Kronecker learner).
- **`gamboostLSS` survival** in R can overflow on heavy-tailed data
  (`WeibullLSS`'s $e^{e}$ gradient); `mboostrs` has the same numeric limit and
  tests use a moderate response scale, matching R.
- Multinomial **newdata/truncated-mstop prediction** is fully supported in R
  for both linear `bols` and smooth `bbs` learners (bols and bbs are the only
  supported learner types for the Multinomial family).

## See also

- [](../appendix/r_parity_matrix) — full feature matrix comparing R and mboostrs support.
- [](testing.ipynb) — how parity tests and finite-difference checks validate the implementation.